# 🎓 **Workshop: Fine-Tuning Gemma for KMU Datensouveränität**

**Mittelstand-Digital Zentrum Spreeland**  
**Dozent:** KI-Trainer-Team  
**Modell:** Gemma 2B Series
**Ziel:** Ein vortrainiertes Open-Weights-Modell auf hochspezifische, interne KMU-Maschinendaten und Wartungsprotokolle anpassen, ohne sensible Daten mit externen Cloud-APIs zu teilen.

---

## 1. Installation & Umgebung einrichten

Wir nutzen **Unsloth**, um das Fine-Tuning auf einer kostenfreien Colab-Grafikkarte (T4 GPU) um den Faktor 2x zu beschleunigen und bis zu 70% Videospeicher (VRAM) einzusparen. Wir verwenden hierbei die offiziell empfohlenen Installationspfade, um Abhängigkeitskonflikte zu vermeiden.

In [ ]:
# Unsloth und offiziell empfohlene Bibliotheken installieren
!pip install --upgrade --force-reinstall "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q
!pip install --no-deps xformers trl peft accelerate bitsandbytes -q

## 2. Hugging Face Authentifizierung (Gemma ist ein lizenziertes Modell)

Gemma ist ein **gated Model** von Google. Um darauf zuzugreifen, müssen Sie den Nutzungsbedingungen einmalig zustimmen:
1. Besuchen Sie **[Hugging Face - Google Gemma 2B IT](https://huggingface.co/google/gemma-2b-it)** und akzeptieren Sie die Nutzungsbedingungen.
2. Erstellen Sie einen kostenfreien Lese-Token (Read Token) unter **[huggingface.co/settings/tokens](https://huggingface.co/settings/tokens)**.
3. Fügen Sie diesen Token in Google Colab links in den **Geheimnissen (Secrets / Schlüssel-Symbol 🔑)** unter dem Namen `HF_TOKEN` ein und aktivieren Sie den Schalter für den Notebook-Zugriff.

## 3. Base Modell laden (quantisiert in 4-Bit)

Wir laden das hocheffiziente Open-Weights-Modell **Gemma 2B IT** in der für Google Colab optimierten pre-quantisierten 4-Bit Version von Unsloth. Dies verhindert zuverlässig Speicherüberläufe auf der T4 GPU.

In [ ]:
from unsloth import FastLanguageModel
import torch
from google.colab import userdata

# Hugging Face Token aus den Colab-Secrets auslesen
try:
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    hf_token = None

# Alternativ hier direkt manuell einsetzen falls keine Secrets genutzt werden:
# hf_token = "hf_YOUR_TOKEN_HERE"

max_seq_length = 2048 # Unterstützt bis zu 8k auf Gemma, aber für die Demo reichen 2048
dtype = None         # Auto-Erkennung (float16 für T4, bfloat16 für moderne GPUs)
load_in_4bit = True  # Spart VRAM durch 4-Bit-Quantisierung

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gemma-2b-it-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    token = hf_token,   # Token wird hier übergeben
)

## 4. LoRA (Low-Rank Adaptation) konfigurieren

Anstatt alle Milliarden Parameter des Modells zu trainieren (was enorm viel VRAM kosten würde), trainieren wir nur einen winzigen Bruchteil an Adapter-Gewichten (LoRA). Das spart Zeit und Rechenleistung.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,               # Rang der Anpassung (empfohlen: 8, 16, 32)
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,     # Optimiert für 0
    bias = "none",        # Optimiert für none
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

## 5. KMU-Spezifischen Datensatz vorbereiten

Wir simulieren den Ernstfall: Interne Wartungsanweisungen und Fehlermeldungen von Werkstatt-Maschinen der fiktiven *Spreeland-Maschinenbau GmbH* werden als Trainingsdatensatz eingelesen und in das offizielle Gemma-Prompt-Format gebracht.

In [ ]:
import pandas as pd
from datasets import Dataset

# Interne Q&A-Datenbank der Spreeland-Maschinenbau GmbH
kmu_data = [
    {
        "instruction": "Fehlermeldung E104 an der Spreeland Press V4 beheben.",
        "response": "Der Fehler E104 an der Press V4 deutet auf einen unzureichenden Öldruck im Hydraulikkreislauf hin. Gehen Sie wie folgt vor:\n1. Schalten Sie die Maschine ab und sichern Sie sie gegen Wiedereinschalten.\n2. Überprüfen Sie den Füllstand des Hydrauliköls am Schauglas (Soll: min. 60%).\n3. Prüfen Sie den Druckminderer Ventil Y3 auf Verstopfung.\n4. Reinigen Sie den Ölfilter F12. Falls der Fehler bestehen bleibt, kontaktieren Sie den Werkskundendienst."
    },
    {
        "instruction": "Wartungsintervall für den Turbomix S300 Mischkopf.",
        "response": "Der Mischkopf des Turbomix S300 erfordert folgende Wartungsintervalle:\n- Täglich: Sichtprüfung auf Dichtheit der Flansche und Reinigung der Mischkammer mit Spülmittellösung Typ M.\n- Alle 500 Betriebsstunden: Nachschmierung der Hauptlagerwelle mit Hochtemperaturfett Spreeland-HT-2.\n- Alle 2.000 Betriebsstunden oder jährlich: Vollständiger Austausch der PTFE-Dichtungen am Mischwellenauslass."
    },
    {
        "instruction": "Wie starte ich den Kalibrierungsmodus der Spreeland LaserCut 9000?",
        "response": "Um den Kalibrierungsmodus an der LaserCut 9000 zu starten:\n1. Navigieren Sie im Hauptmenü des Bedienpanels auf 'Service' -> 'System-Diagnose'.\n2. Geben Sie die KMU-Service-PIN '7739' ein.\n3. Wählen Sie 'Optische Justierung' und drücken Sie den grünen Taster 'Start/OK'.\n4. Die Maschine fährt nun automatisch die 5 Referenzpunkte ab. Platzieren Sie hierzu die Justierplatte auf dem Schneidtisch."
    }
]

df = pd.DataFrame(kmu_data)
dataset = Dataset.from_pandas(df)

# Gemma Chat template definieren
gemma_prompt = """<|turn>user
{}<turn|>
<|turn>model
{}<turn|>"""

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    outputs      = examples["response"]
    texts = []
    for instruction, output in zip(instructions, outputs):
        text = gemma_prompt.format(instruction, output)
        texts.append(text)
    return { "text" : texts, }

dataset = dataset.map(formatting_prompts_func, batched = True)

## 6. Training konfigurieren & starten

Wir füttern die Daten mithilfe des `SFTTrainer` in das Modell. Da unser Datensatz extrem klein ist, genügen 30 bis 60 Trainingsschritte (Steps), um dem Modell dieses Spezialwissen einzuhämmern.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 40,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()

## 7. Verifizierung & Offline Inferenz

Testen wir nun das Modell! Es antwortet nun haargenau mit den geheimen KMU-Handbuch-Spezifikationen, ohne dass diese jemals ins Internet gesendet wurden.

In [ ]:
# Schnelles Inferenz-Backend von Unsloth aktivieren
FastLanguageModel.for_inference(model)

inputs = tokenizer(
[
    gemma_prompt.format(
        "Fehlermeldung E104 an der Spreeland Press V4 beheben.",
        "" # Leer lassen zur Generierung
    )
], return_tensors = "pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 256, use_cache = True)
print(tokenizer.decode(outputs[0]))

## 8. Speichern und Exportieren (z.B. für Ollama / Llama.cpp)

Sie können das trainierte Modell direkt als 16-Bit-Modell, LoRA-Adapter oder im GGUF-Format (für Ollama / LM Studio) exportieren. Damit läuft Ihr feinjustiertes Modell komplett offline im KMU-Betrieb!

In [ ]:
# Speichern der reinen Adapter-Gewichte (nur wenige Megabytes)
model.save_pretrained("spreeland_gemma_adapters")

# Export im GGUF-Format (z.B. Q4_K_M) zur Offline-Nutzung in Ollama
# model.save_pretrained_gguf("gemma-kmu-model", tokenizer, quantization_method = "q4_k_m")